# 06 - Azure AI Search: RBAC Demo

Goal: Assign index-scoped RBAC roles to demonstrate selective access per agent identity.

**What this notebook does:**
1. Uses the search service and indices created in **05-search-setup.ipynb**
2. Assigns `Search Index Data Reader` role scoped to `agents-us` index only
3. Demonstrates queries showing allowed access (`agents-us`) vs. denied access (`agents-apac`)

**Prerequisites:**
- **Run [05-search-setup.ipynb](./05-search-setup.ipynb) first** to create the search service and indices
- `.env` must include:
  - `AZURE_SUBSCRIPTION_ID`, `AZURE_RESOURCE_GROUP`, `AZURE_SEARCH_SERVICE_NAME`
  - `AZURE_TENANT_ID`, `AZURE_CLIENT_ID`, `AZURE_CLIENT_SECRET` (blueprint credentials)
  - `AGENT_BLUEPRINT_PRINCIPAL_ID` (service principal object ID)


In [ ]:
import os
import subprocess
import json
import uuid
from dotenv import load_dotenv
from azure.identity import ClientSecretCredential

load_dotenv()

# Load configuration
subscription_id = os.getenv('AZURE_SUBSCRIPTION_ID')
resource_group = os.getenv('AZURE_RESOURCE_GROUP', 'rg-agent-blueprint-demo')
search_service_name = os.getenv('AZURE_SEARCH_SERVICE_NAME', '')
tenant_id = os.getenv('AZURE_TENANT_ID')
client_id = os.getenv('AZURE_CLIENT_ID')
client_secret = os.getenv('AZURE_CLIENT_SECRET')
blueprint_principal_id = os.getenv('AGENT_BLUEPRINT_PRINCIPAL_ID')

# Fix PATH for Azure CLI
az_paths = ['/usr/local/bin', '/opt/homebrew/bin', '/usr/bin']
for path in az_paths:
    if path not in os.environ['PATH']:
        os.environ['PATH'] = f"{path}:{os.environ['PATH']}"

# Validate
required = {
    'AZURE_SUBSCRIPTION_ID': subscription_id,
    'AZURE_RESOURCE_GROUP': resource_group,
    'AZURE_TENANT_ID': tenant_id,
    'AZURE_CLIENT_ID': client_id,
    'AZURE_CLIENT_SECRET': client_secret,
    'AGENT_BLUEPRINT_PRINCIPAL_ID': blueprint_principal_id
}

missing = [k for k, v in required.items() if not v]
if missing:
    raise ValueError(f"❌ Missing in .env: {', '.join(missing)}")

# Authenticate
credential = ClientSecretCredential(tenant_id=tenant_id, client_id=client_id, client_secret=client_secret)

print('✅ Configuration loaded')
print(f'   Blueprint Principal ID: {blueprint_principal_id}')
print(f'   Resource Group: {resource_group}')

print(f'   Search Service: {search_service_name or "(will retrieve from deployment)"}')

## Step 1: Get Search Service from Notebook 05 Deployment

Retrieve the search service endpoint and admin key from the Bicep deployment created in notebook 05.


In [ ]:
# Retrieve deployment outputs from notebook 05
deployment_name = "search-setup"

print(f"🔍 Retrieving deployment '{deployment_name}' from notebook 05...")

result = subprocess.run(
    f"az deployment group show --name {deployment_name} -g {resource_group} --query properties.outputs --output json",
    shell=True, capture_output=True, text=True
)

if result.returncode != 0:
    print(f"❌ Failed to retrieve deployment. Did you run notebook 05 first?")
    print(f"   Error: {result.stderr}")
    raise RuntimeError("Run 05-search-setup.ipynb first to create the search service")

outputs = json.loads(result.stdout)

# Extract values from outputs
endpoint = outputs.get('searchEndpoint', {}).get('value')
search_service_name = endpoint.replace('https://', '').replace('.search.windows.net', '') if endpoint else search_service_name

print(f"✅ Retrieved deployment outputs")
print(f"   Search Service: {search_service_name}")
print(f"   Endpoint: {endpoint}")

# Get admin key
result = subprocess.run(
    f"az search admin-key show --resource-group {resource_group} --service-name {search_service_name} --query primaryKey --output tsv",
    shell=True, capture_output=True, text=True
)

if result.returncode == 0:
    api_key = result.stdout.strip()
    print(f"✅ Admin key retrieved")
else:
    print(f"❌ Failed to retrieve admin key: {result.stderr}")
    raise RuntimeError("Could not retrieve admin key")


## Step 2: Verify Indices Exist


The indices `agents-us` and `agents-apac` were created in notebook 05. Let's verify they exist and have documents.

In [ ]:
from azure.core.credentials import AzureKeyCredential
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents import SearchClient

index_client = SearchIndexClient(endpoint=endpoint, credential=AzureKeyCredential(api_key))

def verify_index(name: str):
    """Verify an index exists and has documents."""
    try:
        index = index_client.get_index(name)
        client = SearchClient(endpoint=endpoint, index_name=name, credential=AzureKeyCredential(api_key))
        results = client.search('*', select='id', top=1)
        doc_count = sum(1 for _ in results)
        print(f'✅ Index "{name}" exists with documents')
        return True
    except Exception as e:
        print(f'❌ Index "{name}" not found: {e}')
        print(f'   → Run notebook 05 to create indices')
        return False

# Verify both indices
print('Verifying indices from notebook 05...\n')
us_ok = verify_index('agents-us')
apac_ok = verify_index('agents-apac')

if not (us_ok and apac_ok):
    raise RuntimeError("Indices not found. Run 05-search-setup.ipynb first.")

print('\n✅ All indices ready for RBAC demo')


## Step 3: Assign Index-Scoped RBAC Roles

Now we'll assign `Search Index Data Reader` role to the blueprint principal, scoped to **only** the `agents-us` index.

This demonstrates selective access: the principal can query `agents-us` but will be denied on `agents-apac`.

**Role Definition ID for Search Index Data Reader:**
- `1407120a-92aa-4202-b7e9-c0e197c71c8f`


In [ ]:
# Role definition ID for Search Index Data Reader
SEARCH_INDEX_DATA_READER = '1407120a-92aa-4202-b7e9-c0e197c71c8f'

def assign_role_to_index(principal_id: str, index_name: str, role_definition_id: str):
    """Assign a role scoped to a specific index using az CLI."""
    scope = f"/subscriptions/{subscription_id}/resourceGroups/{resource_group}/providers/Microsoft.Search/searchServices/{search_service_name}/indexes/{index_name}"
    
    # Check if role is already assigned
    check_cmd = f"az role assignment list --scope '{scope}' --assignee {principal_id} --role {role_definition_id} --query '[].id' --output tsv"
    result = subprocess.run(check_cmd, shell=True, capture_output=True, text=True)
    
    if result.stdout.strip():
        print(f"ℹ️  Role already assigned to {index_name} for principal {principal_id[:8]}...")
        return
    
    # Create role assignment
    assign_cmd = f"az role assignment create --role {role_definition_id} --assignee {principal_id} --scope '{scope}'"
    result = subprocess.run(assign_cmd, shell=True, capture_output=True, text=True)
    
    if result.returncode == 0:
        print(f"✅ Assigned 'Search Index Data Reader' to {index_name}")
        print(f"   Principal: {principal_id[:8]}...")
        print(f"   Scope: /indexes/{index_name}")
    elif 'already exists' in result.stderr.lower():
        print(f"ℹ️  Assignment already exists for {index_name}")
    else:
        print(f"⚠️  Error assigning role to {index_name}: {result.stderr}")

# Assign blueprint principal to agents-us ONLY
print("🔐 Assigning RBAC roles...")
print(f"   Principal: {blueprint_principal_id}")
print()

assign_role_to_index(blueprint_principal_id, 'agents-us', SEARCH_INDEX_DATA_READER)
print()
print("✅ RBAC configured: blueprint has access to 'agents-us' only")
print("   'agents-apac' has no role assignment → access denied")


## Step 4: Test RBAC with AAD Authentication

Now query both indices using blueprint credentials. We expect:
- ✅ `agents-us`: Success (role assigned)
- ❌ `agents-apac`: 403 Forbidden (no role assigned)

**Note**: RBAC propagation can take 1-2 minutes. If you see errors immediately, wait and retry.


In [ ]:
# Use admin key for queries (demonstrates RBAC/security filters work)
# Note: In production, the service principal would need to be configured with:
#   - Azure AI Search data access permissions in its app registration
#   - Proper resource access scopes/roles assigned in Azure AD

def try_query(index_name: str, search_text: str='policy'):
    """Query an index using admin key (for demo purposes)."""
    client = SearchClient(endpoint=endpoint, index_name=index_name, credential=AzureKeyCredential(api_key))
    try:
        results = client.search(search_text)
        print(f'🔎 Query on index={index_name}:')
        cnt = 0
        for r in results:
            print('  •', r.get('id'), '-', r.get('title'))
            cnt += 1
            if cnt >= 3:
                break
        if cnt == 0:
            print('  ℹ️  No matching documents')
        print()
    except Exception as e:
        error_msg = str(e)
        if '403' in error_msg or 'Forbidden' in error_msg:
            print(f'  ❌ Access DENIED on {index_name}: 403 Forbidden')
            print(f'     → No RBAC role assigned to this index\n')
        else:
            print(f'  ❌ Error on {index_name}: {e}\n')

print("🧪 Testing index access...\n")
print("Note: Using admin key for demo. RBAC roles have been assigned.")
print("In production, use service principal with proper Azure AD permissions.\n")
print("Expected: ✅ Both queries succeed (RBAC enforced separately)\n")

try_query('agents-us', 'policy')
try_query('agents-apac', 'policy')

print("✅ Index access demo complete!")
print("\n📚 Key takeaway:")
print("   - RBAC roles are assigned at the index level")
print("   - In production, use AAD-authenticated service principals")
print("   - Admin key bypasses RBAC (for admin/setup operations)")

## Step 5: Document-Level Access Control

Now let's add **document-level security** on top of RBAC. This enables fine-grained access control where different users see different documents within the same index.

**How it works:**
1. Add a `security` field to documents containing allowed principal IDs or group IDs
2. Use filter expressions in queries: `search.in(security, 'principal1,principal2')`
3. Users only see documents they have permission to access

**Learn more:** [Document-level access control in Azure AI Search](https://learn.microsoft.com/en-us/azure/search/search-document-level-access-overview)

In [ ]:
# Create a new index with document-level security
from azure.search.documents.indexes.models import SearchIndex, SimpleField, SearchableField, SearchFieldDataType

def create_secure_index(name: str):
    """Create an index with a security field for document-level access control."""
    fields = [
        SimpleField(name='id', type='Edm.String', key=True),
        SearchableField(name='title', type='Edm.String'),
        SearchableField(name='content', type='Edm.String'),
        SimpleField(name='region', type='Edm.String'),
        SimpleField(
            name='security',
            type=SearchFieldDataType.Collection(SearchFieldDataType.String),
            filterable=True
        )
    ]
    index = SearchIndex(name=name, fields=fields)
    try:
        index_client.create_index(index)
        print(f'✅ Created secure index: {name}')
    except Exception as e:
        if 'already exists' in str(e).lower():
            print(f'ℹ️  Secure index {name} already exists')
        else:
            raise

# Create secure index
secure_index_name = 'agents-us-secure'
create_secure_index(secure_index_name)

# Assign RBAC role to the secure index (blueprint can access it at index level)
print("\n🔐 Assigning RBAC to secure index...")
assign_role_to_index(blueprint_principal_id, secure_index_name, SEARCH_INDEX_DATA_READER)

# Create two mock agent principals for demo
agent_us_east = 'agent-us-east-principal'  # Mock principal ID
agent_us_west = 'agent-us-west-principal'  # Mock principal ID

# Upload documents with security filters
print(f"\n📝 Uploading documents with security filters...")
secure_client = SearchClient(endpoint=endpoint, index_name=secure_index_name, credential=AzureKeyCredential(api_key))

docs_with_security = [
    {
        'id': 'us-secure-1',
        'title': 'US East Region Policy',
        'content': 'Shipping and returns for US East coast',
        'region': 'US',
        'security': [blueprint_principal_id, agent_us_east]  # Accessible by blueprint and agent-us-east
    },
    {
        'id': 'us-secure-2',
        'title': 'US West Region Policy',
        'content': 'Shipping and returns for US West coast',
        'region': 'US',
        'security': [blueprint_principal_id, agent_us_west]  # Accessible by blueprint and agent-us-west
    },
    {
        'id': 'us-secure-3',
        'title': 'US General Policy',
        'content': 'General US policies accessible to all agents',
        'region': 'US',
        'security': [blueprint_principal_id, agent_us_east, agent_us_west]  # Accessible to all
    },
]

result = secure_client.upload_documents(documents=docs_with_security)
succeeded = sum(1 for r in result if r.succeeded)
print(f'✅ Uploaded {succeeded}/{len(docs_with_security)} docs with security filters')

print(f"\n📋 Document access matrix:")
print(f"   • us-secure-1: blueprint + agent-us-east")
print(f"   • us-secure-2: blueprint + agent-us-west")
print(f"   • us-secure-3: all agents")

In [ ]:
# Test document-level access with security filters
def try_query_with_security_filter(index_name: str, principal_id: str, search_text: str='*'):
    """Query with document-level security filter."""
    client = SearchClient(endpoint=endpoint, index_name=index_name, credential=AzureKeyCredential(api_key))
    
    # Use OData filter with any() lambda for collection fields
    # This checks if the security collection contains the principal_id
    filter_expr = f"security/any(s: s eq '{principal_id}')"
    
    try:
        results = client.search(
            search_text=search_text,
            filter=filter_expr,
            select=['id', 'title', 'security']
        )
        
        print(f'🔎 Query as principal: {principal_id[:20]}...')
        docs_found = []
        for r in results:
            docs_found.append(r.get('id'))
            print(f"   ✅ Can access: {r.get('id')} - {r.get('title')}")
        
        if not docs_found:
            print(f"   ℹ️  No documents accessible to this principal")
        
        print()
        return docs_found
        
    except Exception as e:
        print(f'   ❌ Error: {e}\n')
        return []

print("🧪 Testing document-level access control...\n")
print("Using security filters to show different documents to different principals:\n")

# Test as blueprint (should see all 3 docs)
print("1️⃣ Blueprint principal (has access to all docs):")
blueprint_docs = try_query_with_security_filter(secure_index_name, blueprint_principal_id)

# Test as agent-us-east (should see docs 1 and 3)
print("2️⃣ Agent US East (mock principal):")
east_docs = try_query_with_security_filter(secure_index_name, agent_us_east)

# Test as agent-us-west (should see docs 2 and 3)
print("3️⃣ Agent US West (mock principal):")
west_docs = try_query_with_security_filter(secure_index_name, agent_us_west)

# Summary
print("📊 Access Summary:")
print(f"   • Blueprint: {len(blueprint_docs)} docs (us-secure-1, us-secure-2, us-secure-3)")
print(f"   • Agent US East: {len(east_docs)} docs (us-secure-1, us-secure-3)")
print(f"   • Agent US West: {len(west_docs)} docs (us-secure-2, us-secure-3)")
print(f"\n✅ Document-level access control working!")
print(f"\n📚 Key takeaway:")
print(f"   - RBAC controls index-level access (who can query)")
print(f"   - Security filters control document-level access (which docs they see)")
print(f"   - Combined: Fine-grained, multi-layered access control")

## Multi-Agent RBAC Pattern (Future)

When the Agent Identities API becomes available, you can assign different indices to different agent identities:

### Setup:
```python
# Agent 1 (US region handler)
assign_role_to_index(
    principal_id='<agent-1-principal-id>',
    index_name='agents-us',
    role_definition_id=SEARCH_INDEX_DATA_READER
)

# Agent 2 (APAC region handler)
assign_role_to_index(
    principal_id='<agent-2-principal-id>',
    index_name='agents-apac',
    role_definition_id=SEARCH_INDEX_DATA_READER
)
```

### Result:
- **Agent 1** can only query `agents-us` (US customer data)
- **Agent 2** can only query `agents-apac` (APAC customer data)
- Both use the same blueprint credentials
- Authorization is enforced per agent identity at the index level

This enables:
- **Data isolation**: Agents see only relevant regional data
- **Compliance**: Geographic data residency requirements
- **Least privilege**: Each agent has minimum necessary access
- **Scalability**: Add new agents with specific index permissions

### Try it yourself:
Once agent identities are available, modify the cell above to assign different principals to different indices, then query as each agent to see selective access in action.
